<a href="https://colab.research.google.com/gist/lanmower/50261087154f544807eed00872f95d5e/colab_round.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# traintai — one training round (Colab, TPU v5e)

Runs one full training round of the tai NPC dialog model end to end via
`src/round.py`: **prepare → SFT top-up → GRPO → forge → sim_eval**, starting
from the ship checkpoint `ple-st-r16-grpo.pt` (honest forge pass 74%).

**Before you start: Runtime → Change runtime type → TPU v5e.**
Every round-path script auto-detects the TPU through `src/device.py`
(XLA TPU → CUDA → MPS → CPU), so no code flags are needed.

Round state (see `AGENTS.md` for the measured basis):

- `r16` is the ship checkpoint. Rounds `r17`–`r22` (GRPO action-reward
  shaping, four variants) are all measured **non-ship** — reward shaping on
  actions is a dead lever at this scale.
- The next round is **`st-r23`**. The open levers are data-side:
  rejection-sample the model's own VALID oracle-matching actions into the
  flywheel, denser exact-action sim SFT rows, and multi-sentence gold chains
  for chain depth. Keep `npc_grpo.py --action-reward` at its default `off`
  (= the r16 ship recipe).

No tokens or secrets are needed — every download below is anonymous.
Never paste credentials into a notebook (a leaked HF token in a gist is a
documented incident in this project's history).

TPU notes: the first training steps are slow while XLA compiles the step
graph, and generation stages (GRPO rollouts, forge) recompile per sequence
shape — wall-clock is dominated by compile cache warm-up, not a hang.

In [15]:
import torch
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    device = xm.xla_device()
    hw_type = f"TPU ({xr.device_type()})"
except ImportError:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        hw_type = "GPU (CUDA)"
    else:
        device = torch.device("cpu")
        hw_type = "CPU"

print(f"Detected hardware: {hw_type} | Device: {device}")
if hw_type == "CPU":
    print("Warning: Running on CPU will be extremely slow for training.")

Detected hardware: GPU (CUDA) | Device: cuda


## Clone and environment

Fresh clone, then a **Python 3.10** venv carrying the stack that is
empirically working on Colab's TPU v5e: `torch==2.4.0` plus the matching
`torch_xla-2.4.0` tpuvm wheel from the pytorch-xla GCS bucket (newer
torch_xla releases do not bind to this runtime). `uv sync` is skipped on
purpose: the lockfile targets Python ≥3.12 and a CPU-only torch ≥2.13,
so the venv is populated directly with `uv pip install --python`.

In [16]:
import os
%cd /content
if not os.path.exists("traintai"):
    !git clone https://github.com/AnEntrypoint/traintai.git
else:
    !git -C traintai pull --ff-only
%cd traintai

/content
Already up to date.
/content/traintai


In [17]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
# Clean up previous venv attempts to avoid path confusion
!rm -rf .venv
# Install missing dependencies directly to system-site (user) to maintain TPU stack access
!uv pip install --system numpy==1.26.4 requests tokenizers tqdm

downloading uv 0.12.1 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.13 environment at: /usr
Resolved 20 packages in 353ms
Prepared 1 package in 548ms
Uninstalled 1 package in 48ms
Installed 1 package in 17ms
 - numpy==2.0.2
 + numpy==1.26.4


In [18]:
# Verify the stack based on detected hardware
import torch
print(f"Torch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA available: {torch.version.cuda} | Device: {torch.cuda.get_device_name(0)}")
else:
    try:
        import torch_xla.runtime as xr
        print(f"TPU Runtime: {xr.device_type()}")
    except ImportError:
        print("Running on standard CPU stack.")

Torch version: 2.11.0+cu128
CUDA available: 12.8 | Device: Tesla T4


## Ship checkpoint

The r16 ship checkpoint is pulled from the GitHub release (anonymous, no
token). Every round starts from it — never from a non-ship round.

In [19]:
!mkdir -p runs
!mkdir -p src/../runs
!mkdir -p /content/traintai/runs
!curl -sL -o runs/ple-st-r16-grpo.pt https://github.com/AnEntrypoint/traintai/releases/download/v0.1.0/ple-st-r16-grpo.pt
!ls -la runs/

total 112808
drwxr-xr-x 2 root root      4096 Aug  4 14:44 .
drwxr-xr-x 8 root root      4096 Aug  4 14:44 ..
-rw-r--r-- 1 root root 115504792 Aug  4 14:44 ple-st-r16-grpo.pt


## Run the round

One round = prepare (builds the TinyStories token bins itself on first run,
~300MB anonymous HF download) → 300 SFT steps → 200 GRPO steps → 720-rollout
forge dashboard → held-out sim_eval. Each stage logs to `runs/<tag>*.log` and
a summary block prints at the end. Every stage runs under the 3.10 venv's
python (`round.py` re-enters itself via `sys.executable`, so the venv
propagates). Run ONE round at a time.

In [20]:
TAG = "st-r23"  # next round; r16 ships, r17–r22 measured non-ship

In [ ]:
# Ensure the directory exists as an absolute path and run training
import os
TAG = "st-r23"
os.environ['TAG'] = TAG

# Create the absolute path to 'runs' to avoid relative path errors
!mkdir -p /content/traintai/runs

# Run using system python
print(f"Starting training round: {TAG}...")
!python3 src/round.py --prev runs/ple-st-r16-grpo.pt --tag $TAG

Starting training round: st-r23...


## Holdout generalization gate

Teacher-forced perplexity on the PIPPA holdout (never in the training bins).
Reference points: r16 ship = 358.74, r19 = 357.11. A large jump means the
round overfit the real data — do not ship it.

In [ ]:
import os
# Re-verify directory presence
if not os.path.exists("/content/traintai"):
    print("Error: Project directory /content/traintai missing. Please run the 'Clone and environment' cell.")
else:
    %cd /content/traintai
    TAG = "st-r23"
    checkpoint_path = f"/content/traintai/runs/ple-{TAG}-grpo.pt"
    eval_script = "/content/traintai/src/holdout_eval.py"

    if os.path.exists(checkpoint_path):
        !python3 {eval_script} {checkpoint_path}
    else:
        print(f"Error: Checkpoint {checkpoint_path} not found. You must run the training round cell (hhjg5XS7Ixxg) first!")

## Keep the artifacts

Colab runtimes are ephemeral — download the checkpoint and stage logs, then
record the round's measured results in `AGENTS.md` (a result lands once,
with its measurement).

In [ ]:
# Package and download logs and checkpoint if they exist
import os
TAG = "st-r23"

if os.path.exists(f"runs/ple-{TAG}-grpo.pt"):
    !tar -czf {TAG}-artifacts.tar.gz runs/ple-{TAG}-grpo.pt runs/{TAG}*.log
    from google.colab import files
    files.download(f"{TAG}-artifacts.tar.gz")
else:
    print(f"Error: Checkpoint runs/ple-{TAG}-grpo.pt not found. Did the training round finish?")